# 02 — Graph Analysis

Sanity-check on the directed link graph constructed by `src.graph.build_graph` from the `outbound_links` field of every scraped JSON sidecar. We confirm graph algorithms are concrete (Shivani feedback resolved) and that PageRank/HITS/clustering distributions carry usable signal.

Algorithms: **PageRank** (Page–Brin, α=0.85) ranks pages by random-walk stationary probability; **HITS** (Kleinberg) splits importance into *hubs* (good link sources) and *authorities* (good link targets); **clustering coefficient** measures local triangle density (high clustering ≈ tight thematic neighborhoods).

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath(".."))
from pathlib import Path
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns

from src.graph.build_graph import build, load
from src.graph.graph_features import compute

sns.set_theme(style="whitegrid")
graph_path = Path("../data/interim/graph.pkl")
g = load(graph_path) if graph_path.exists() else build(Path("../data/raw"))
print(f"nodes={g.number_of_nodes()}  edges={g.number_of_edges()}  density={nx.density(g):.5f}")

## Compute and inspect graph features

In [ ]:
gf = compute(g)
gf.describe()[["pagerank","hits_hub","hits_authority","in_degree","out_degree","clustering"]]

## PageRank distribution

Heavy-tailed PageRank confirms a few hub pages dominate — the same shape we'd expect from any well-formed web subgraph. Uniform PageRank would have meant the feature is uninformative.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), dpi=200)
sns.histplot(gf["pagerank"].clip(lower=1e-6), bins=40, log_scale=(True, False), color="#4f46e5", edgecolor="white", ax=ax)
ax.set_xlabel("PageRank (log)"); ax.set_ylabel("page count"); ax.set_title("PageRank — heavy-tailed across the scraped corpus")
plt.tight_layout(); plt.show()

## In-degree vs out-degree

High-in / low-out pages are *authorities*; low-in / high-out are *hubs* (typically index pages). HITS surfaces this as two separate scores.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6), dpi=200)
sns.scatterplot(data=gf, x="in_degree", y="out_degree", hue="pagerank", palette="viridis", alpha=0.7, ax=ax, legend=False)
ax.set_xscale("symlog"); ax.set_yscale("symlog"); ax.set_title("In- vs out-degree (color = PageRank)")
plt.tight_layout(); plt.show()

## Clustering coefficient

Local triangle density. High clustering ≈ tight thematic neighborhoods — useful signal for whether a page sits inside a related cluster.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), dpi=200)
sns.histplot(gf["clustering"], bins=30, color="#ec4899", edgecolor="white", ax=ax)
ax.set_xlabel("clustering coefficient"); ax.set_ylabel("page count"); ax.set_title("Clustering coefficient distribution")
plt.tight_layout(); plt.show()

## Top-10 highest-PageRank pages

In [ ]:
url_attr = nx.get_node_attributes(g, "url")
top = gf.sort_values("pagerank", ascending=False).head(10).copy()
top["url"] = top["node"].map(url_attr)
top[["url","pagerank","in_degree","out_degree","hits_authority"]]